In [3]:
# ===================== WhoProfits scraper (single Colab cell) =====================
!pip -q install lxml

import sqlite3, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

# ----------------------------- Settings (edit these) -----------------------------
RUN_LISTING   = True     # phase 1: scrape the company listing
RUN_ENRICH    = True     # phase 2: scrape each company's detail page
REFETCH_ALL   = False    # True = re-fetch every detail page; False = only ones not done yet
WORKERS       = 4        # concurrent detail-page fetches
MIN_INTERVAL  = 0.4      # min seconds between request starts (all threads combined)

DB_PATH            = "whoprofits.db"
BASE_URL           = "https://www.whoprofits.org"
LISTING_URL        = f"{BASE_URL}/companies/all"
TOTAL_EXPECTED     = 472     # only used for a sanity warning
MAX_LISTING_PAGES  = 200     # safety stop
COMMIT_EVERY       = 25

HEADERS = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36")}

ENRICH_COLUMNS = [
    ("company_name_he_detail", "TEXT"), ("address", "TEXT"), ("website", "TEXT"),
    ("description", "TEXT"), ("sector", "TEXT"), ("revenue", "TEXT"), ("region", "TEXT"),
    ("settlements", "TEXT"), ("ownership", "TEXT"), ("partners", "TEXT"),
    ("subsidiaries", "TEXT"), ("valid_until", "TEXT"),
    ("detail_scraped_at", "TEXT"),   # marks finished rows so reruns resume instead of redoing
]

# ----------------------------- HTTP -----------------------------
class RateLimiter:
    """Thread-safe: at least `interval` seconds between request starts."""
    def __init__(self, interval):
        self.interval, self.lock, self.next_time = interval, threading.Lock(), 0.0
    def wait(self):
        with self.lock:
            now = time.monotonic()
            delay = max(0.0, self.next_time - now)
            self.next_time = max(now, self.next_time) + self.interval
        if delay:
            time.sleep(delay)

def make_session(pool_size=10):
    s = requests.Session()
    s.headers.update(HEADERS)
    retry = Retry(total=4, backoff_factor=1.0, status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",), respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry, pool_connections=pool_size, pool_maxsize=pool_size)
    s.mount("https://", adapter); s.mount("http://", adapter)
    return s

SESSION, LIMITER = make_session(), RateLimiter(MIN_INTERVAL)

def fetch(url):
    LIMITER.wait()
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    return r.text

try:
    import lxml; PARSER = "lxml"
except ImportError:
    PARSER = "html.parser"

# ----------------------------- DB -----------------------------
def init_db(path=DB_PATH):
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS companies (
            id INTEGER PRIMARY KEY, company_name TEXT, company_name_he TEXT, company_url TEXT,
            traded_in TEXT, traded_in_url TEXT, headquarters TEXT, headquarters_url TEXT,
            involvement TEXT, involvement_urls TEXT)
    """)
    existing = {r[1] for r in conn.execute("PRAGMA table_info(companies)")}
    for name, typ in ENRICH_COLUMNS:
        if name not in existing:
            conn.execute(f"ALTER TABLE companies ADD COLUMN {name} {typ}")
    conn.commit()
    return conn

# ----------------------------- Phase 1: listing -----------------------------
def _link_info(td):
    a = td.find("a")
    if a:
        href = a.get("href")
        return a.get_text(strip=True), (urljoin(BASE_URL, href) if href else None)
    return td.get_text(strip=True), None

def parse_listing(html):
    soup = BeautifulSoup(html, PARSER)
    tbody = soup.select_one("table tbody")
    if not tbody:
        return []
    rows = []
    for tr in tbody.find_all("tr", recursive=False):
        tds = tr.find_all("td", recursive=False)
        if len(tds) < 5:
            continue
        entry_id = tds[0].get_text(strip=True)
        if not entry_id.isdigit():
            continue

        name_link = tds[1].find("a")
        company_url = urljoin(BASE_URL, name_link["href"]) if name_link and name_link.get("href") else None
        company_name = company_name_he = None
        if name_link:
            he_span = name_link.find("span")
            if he_span:
                company_name_he = he_span.get_text(strip=True)
                he_span.extract()
            company_name = name_link.get_text(strip=True)

        traded_in, traded_in_url = _link_info(tds[2])
        headquarters, hq_url = _link_info(tds[3])
        inv_links = tds[4].find_all("a")

        rows.append({
            "id": int(entry_id), "company_name": company_name, "company_name_he": company_name_he,
            "company_url": company_url, "traded_in": traded_in, "traded_in_url": traded_in_url,
            "headquarters": headquarters, "headquarters_url": hq_url,
            "involvement": " | ".join(a.get_text(strip=True) for a in inv_links),
            "involvement_urls": " | ".join(urljoin(BASE_URL, a["href"]) for a in inv_links if a.get("href")),
        })
    return rows

def save_listing_rows(conn, rows):
    # Upsert (not INSERT OR REPLACE) so re-running never wipes enriched columns
    conn.executemany("""
        INSERT INTO companies (id, company_name, company_name_he, company_url, traded_in, traded_in_url,
                               headquarters, headquarters_url, involvement, involvement_urls)
        VALUES (:id, :company_name, :company_name_he, :company_url, :traded_in, :traded_in_url,
                :headquarters, :headquarters_url, :involvement, :involvement_urls)
        ON CONFLICT(id) DO UPDATE SET
            company_name=excluded.company_name, company_name_he=excluded.company_name_he,
            company_url=excluded.company_url, traded_in=excluded.traded_in,
            traded_in_url=excluded.traded_in_url, headquarters=excluded.headquarters,
            headquarters_url=excluded.headquarters_url, involvement=excluded.involvement,
            involvement_urls=excluded.involvement_urls
    """, rows)
    conn.commit()

def scrape_listing(conn):
    print("=== Phase 1: scraping listing ===")
    seen = set()
    for page in range(1, MAX_LISTING_PAGES + 1):
        url = LISTING_URL if page == 1 else f"{LISTING_URL}?page={page}"
        try:
            rows = parse_listing(fetch(url))
        except Exception as e:
            print(f"  page {page} failed: {e}")
            break
        new_rows = [r for r in rows if r["id"] not in seen]
        if not new_rows:
            print(f"  page {page}: nothing new - end of pagination.")
            break
        save_listing_rows(conn, new_rows)
        seen.update(r["id"] for r in new_rows)
        print(f"  page {page}: +{len(new_rows)} (total {len(seen)})")
    if len(seen) != TOTAL_EXPECTED:
        print(f"  Note: got {len(seen)} rows, expected {TOTAL_EXPECTED}. Check pagination if this looks wrong.")
    print(f"Phase 1 done: {len(seen)} listing rows.\n")

# ----------------------------- Phase 2: detail pages -----------------------------
def get_sidebar_field(soup, title):
    """<h5 class='comp-left-bar-title'>Title</h5> -> text of the <a> tags in the next <p>."""
    h5 = soup.find("h5", class_="comp-left-bar-title", string=lambda s: s and s.strip() == title)
    if not h5:
        return None
    p = h5.find_next_sibling("p")
    if not p:
        return None
    links = p.find_all("a")
    if links:
        return " | ".join(a.get_text(strip=True) for a in links)
    return p.get_text(" ", strip=True) or None

def get_block_by_title(soup, title, block_class="comp-right-bar-block"):
    """Ownership / Partners / Subsidiaries block; prefers the full-text hidden `_print` copy."""
    candidates = [
        b for b in soup.find_all("div", class_=block_class)
        if (h5 := b.find("h5", class_="comp-right-bar-title")) and h5.get_text(strip=True) == title
    ]
    if not candidates:
        return None
    block = next((b for b in candidates if b.get("id", "").endswith("_print")), candidates[0])
    parts = []
    for p in block.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text and not text.startswith("* This section refers to"):
            parts.append(text)
    return " ".join(parts) if parts else None

def parse_detail(html):
    soup = BeautifulSoup(html, PARSER)

    he_tag   = soup.find("h5", class_="comp-middle-bar-title2")
    addr_tag = soup.find("p",  class_="comp-middle-bar-address")

    website = None
    title_block = soup.find("div", class_="compNewTitle")
    if title_block:
        for a in title_block.find_all("a", href=True):
            if a["href"].startswith("http") and "whoprofits.org" not in a["href"]:
                website = a["href"]
                break

    description = None
    desc_block = soup.find("div", class_="comp-middle-bar-text")
    if desc_block:
        paras = [p.get_text(" ", strip=True) for p in desc_block.find_all("p")]
        description = "\n\n".join(p for p in paras if p) or None

    valid_until = None
    for p in soup.find_all("p", style=True):
        if "All the information contained" in p.get_text():
            valid_until = p.get_text(strip=True)
            break

    return {
        "company_name_he_detail": he_tag.get_text(strip=True) if he_tag else None,
        "address": addr_tag.get_text(strip=True) if addr_tag else None,
        "website": website,
        "description": description,
        "sector": get_sidebar_field(soup, "Sector"),
        "revenue": get_sidebar_field(soup, "Revenue"),
        "region": get_sidebar_field(soup, "Region"),
        "settlements": get_sidebar_field(soup, "Settlements"),
        "ownership": get_block_by_title(soup, "Ownership"),
        "partners": get_block_by_title(soup, "Partners"),
        "subsidiaries": get_block_by_title(soup, "Subsidiaries"),
        "valid_until": valid_until,
        # overwrite listing values only when the detail page has them
        "involvement": get_sidebar_field(soup, "Involvement"),
        "traded_in": get_sidebar_field(soup, "Traded"),
        "headquarters": get_sidebar_field(soup, "Headquarters"),
    }

def _work(row_id, url):
    """Worker thread: network + parsing only, no DB access."""
    data = parse_detail(fetch(url))
    data["id"] = row_id
    return data

UPDATE_SQL = """
    UPDATE companies SET
        company_name_he_detail=:company_name_he_detail, address=:address, website=:website,
        description=:description, sector=:sector, revenue=:revenue, region=:region,
        settlements=:settlements, ownership=:ownership, partners=:partners,
        subsidiaries=:subsidiaries, valid_until=:valid_until,
        involvement  = COALESCE(:involvement, involvement),
        traded_in    = COALESCE(:traded_in, traded_in),
        headquarters = COALESCE(:headquarters, headquarters),
        detail_scraped_at = datetime('now')
    WHERE id = :id
"""

def scrape_details(conn, refetch_all=False, workers=WORKERS):
    print("=== Phase 2: enriching from detail pages ===")
    where = "company_url IS NOT NULL" + ("" if refetch_all else " AND detail_scraped_at IS NULL")
    todo = conn.execute(f"SELECT id, company_url FROM companies WHERE {where} ORDER BY id").fetchall()
    print(f"{len(todo)} companies to enrich ({workers} workers).")

    failed, done = [], 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(_work, rid, url): (rid, url) for rid, url in todo}
        for fut in tqdm(as_completed(futures), total=len(futures)):
            rid, url = futures[fut]
            try:
                conn.execute(UPDATE_SQL, fut.result())
            except Exception as e:
                failed.append((rid, url, str(e)))
                continue
            done += 1
            if done % COMMIT_EVERY == 0:
                conn.commit()
    conn.commit()

    print(f"Phase 2 done: {done} enriched, {len(failed)} failed.")
    for rid, url, err in failed:
        print(f"  FAILED id={rid} {url} -> {err}")
    if failed:
        print("Re-run this cell to retry only the failed/missing ones.")

# ----------------------------- Run -----------------------------
conn = init_db()
try:
    if RUN_LISTING:
        scrape_listing(conn)
    if RUN_ENRICH:
        scrape_details(conn, refetch_all=REFETCH_ALL)
finally:
    conn.close()
print("All done.")

=== Phase 1: scraping listing ===
  page 1: +20 (total 20)
  page 2: +20 (total 40)
  page 3: +20 (total 60)
  page 4: +20 (total 80)
  page 5: +20 (total 100)
  page 6: +20 (total 120)
  page 7: +20 (total 140)
  page 8: +20 (total 160)
  page 9: +20 (total 180)
  page 10: +20 (total 200)
  page 11: +20 (total 220)
  page 12: +20 (total 240)
  page 13: +20 (total 260)
  page 14: +20 (total 280)
  page 15: +20 (total 300)
  page 16: +20 (total 320)
  page 17: +20 (total 340)
  page 18: +20 (total 360)
  page 19: +20 (total 380)
  page 20: +20 (total 400)
  page 21: +20 (total 420)
  page 22: +20 (total 440)
  page 23: +20 (total 460)
  page 24: +12 (total 472)
  page 25: nothing new - end of pagination.
Phase 1 done: 472 listing rows.

=== Phase 2: enriching from detail pages ===
472 companies to enrich (4 workers).


  0%|          | 0/472 [00:00<?, ?it/s]

Phase 2 done: 472 enriched, 0 failed.
All done.


In [5]:
import sqlite3, pandas as pd
from google.colab import files

with sqlite3.connect("whoprofits.db") as c:
    df = pd.read_sql("SELECT * FROM companies ORDER BY id", c)

print(df.shape)                       # should be ~472 rows x 23 columns
print(df.columns.tolist())            # the last ones should be ...valid_until, detail_scraped_at
df.to_csv("whoprofits.csv", index=False, encoding="utf-8-sig")

(472, 23)
['id', 'company_name', 'company_name_he', 'company_url', 'traded_in', 'traded_in_url', 'headquarters', 'headquarters_url', 'involvement', 'involvement_urls', 'company_name_he_detail', 'address', 'website', 'description', 'sector', 'revenue', 'region', 'settlements', 'ownership', 'partners', 'subsidiaries', 'valid_until', 'detail_scraped_at']
